<a href="https://colab.research.google.com/github/hemachandranrcs24-spec/Ai-Lab-5H/blob/main/1BM24CS115_WEEK8_FOL_Forward_Chaining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import re

# Check whether a term is a variable
def is_variable(x):
    return x.islower()

# Parse a predicate
def parse_atom(atom):
    atom = atom.strip()
    match = re.match(r'(\w+)\(?([^\)]*)\)?', atom)

    if not match:
        raise ValueError("Invalid predicate: " + atom)

    name = match.group(1)
    args = [x.strip() for x in match.group(2).split(",") if x.strip()]

    return name, args


# Match a rule condition with a fact
def unify(pattern, fact, substitution):
    p_name, p_args = parse_atom(pattern)
    f_name, f_args = parse_atom(fact)

    if p_name != f_name or len(p_args) != len(f_args):
        return None

    new_sub = substitution.copy()

    for p, f in zip(p_args, f_args):

        if is_variable(p):
            if p in new_sub:
                if new_sub[p] != f:
                    return None
            else:
                new_sub[p] = f

        elif p != f:
            return None

    return new_sub


# Replace variables with actual values
def substitute(atom, sub):
    name, args = parse_atom(atom)

    new_args = [sub.get(x, x) for x in args]

    return name + "(" + ",".join(new_args) + ")"


# Forward chaining algorithm
def forward_chaining(facts, rules, query):

    known = set(facts)

    print("\nInitial Facts:")
    for f in sorted(known):
        print(f)

    iteration = 1

    while True:

        new_facts = set()

        for premises, conclusion in rules:

            # Start with an empty substitution
            substitutions = [{}]

            # Match every premise against known facts
            for premise in premises:

                updated = []

                for sub in substitutions:
                    for fact in known:
                        result = unify(premise, fact, sub)

                        if result is not None:
                            updated.append(result)

                substitutions = updated

                if not substitutions:
                    break

            # Generate conclusion
            for sub in substitutions:
                inferred = substitute(conclusion, sub)

                if inferred not in known:
                    new_facts.add(inferred)

        if not new_facts:
            break

        print("\nIteration", iteration)

        for f in sorted(new_facts):
            print("New Fact:", f)

        known.update(new_facts)
        iteration += 1

    print("\nFinal Knowledge Base:")
    for f in sorted(known):
        print(f)

    print("\nQuery:", query)

    if query in known:
        print("RESULT: QUERY PROVED (TRUE)")
    else:
        print("RESULT: QUERY NOT PROVED (FALSE)")


# ---------------- MAIN PROGRAM ----------------

print("FIRST ORDER LOGIC - FORWARD CHAINING")

# Enter facts
n = int(input("\nEnter number of facts: "))
facts = []

print("Enter facts (Example: Parent(John,Alice))")

for i in range(n):
    facts.append(input("Fact " + str(i+1) + ": ").strip())

# Enter rules
r = int(input("\nEnter number of rules: "))
rules = []

print("\nEnter rules using -> and &")
print("Example: Parent(x,y) & Ancestor(y,z) -> Ancestor(x,z)")

for i in range(r):

    rule = input("Rule " + str(i+1) + ": ")

    left, right = rule.split("->")

    premises = [x.strip() for x in left.split("&")]
    conclusion = right.strip()

    rules.append((premises, conclusion))

# Enter query
query = input("\nEnter query to prove: ").strip()

# Execute
forward_chaining(facts, rules, query)

FIRST ORDER LOGIC - FORWARD CHAINING

Enter number of facts: 4
Enter facts (Example: Parent(John,Alice))
Fact 1:  Enemy(A,America)
Fact 2: Owns(A,T1)
Fact 3: Missile(T1)
Fact 4: American(Robert)

Enter number of rules: 4

Enter rules using -> and &
Example: Parent(x,y) & Ancestor(y,z) -> Ancestor(x,z)
Rule 1: American(x) & Weapon(y) & Sells(x,y,z) & Hostile(z) -> Criminal(x)
Rule 2:  Missile(x) & Owns(A,x) -> Sells(Robert,x,A)
Rule 3:  Missile(x) -> Weapon(x)
Rule 4:  Enemy(x,America) -> Hostile(x)

Enter query to prove:  Criminal(Robert)

Initial Facts:
American(Robert)
Enemy(A,America)
Missile(T1)
Owns(A,T1)

Iteration 1
New Fact: Hostile(A)
New Fact: Sells(Robert,T1,A)
New Fact: Weapon(T1)

Iteration 2
New Fact: Criminal(Robert)

Final Knowledge Base:
American(Robert)
Criminal(Robert)
Enemy(A,America)
Hostile(A)
Missile(T1)
Owns(A,T1)
Sells(Robert,T1,A)
Weapon(T1)

Query: Criminal(Robert)
RESULT: QUERY PROVED (TRUE)
